# 📈 Sistema de Trading Automatizado v9.0 – GEBRA Portfolio (COMPLETO)
**Otimizações:** Brapi.dev, cache, vetorização, Google Sheets, Telegram, relatório detalhado por e‑mail.

In [ ]:
import os
EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')
PARAMS_BAIXA_VOL = {
    'kelly_frac': 0.30, 'wyckoff_threshold': 0.75, 'gap_max_pct': 0.055,
    'custos_pct': 0.003, 'exigir_volume_anormal': False,
    'risco_percentual_maximo': 0.15, 'preco_minimo': 2.00
}
PARAMS_ALTA_VOL = {
    'kelly_frac': 0.15, 'wyckoff_threshold': 0.85, 'gap_max_pct': 0.03,
    'custos_pct': 0.006, 'exigir_volume_anormal': True,
    'risco_percentual_maximo': 0.10, 'preco_minimo': 2.00
}
PARAMS_ATIVOS = PARAMS_BAIXA_VOL.copy()
MAX_SETUPS_POR_DIA = 5
MAX_PERDAS_CONSECUTIVAS = 3
DRAWDOWN_MAX_DIARIO = 0.02
MAX_DIAS_LOG = 30
HABILITAR_LOGGING = True
ARQUIVO_LOG = "trading_log_v85.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v85.log"
CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0
SETORES_BLOQUEADOS = ['AEREA']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA', 'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']
FALLBACK_TICKERS = ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'SUZB3', 'GGBR4', 'MGLU3', 'VVAR3', 'RENT3', 'RAIL3', 'CCRO3', 'ELET3', 'CPFE3', 'SBSP3', 'SANB11', 'B3SA3', 'JBSS3', 'BRFS3', 'KLBN11', 'EQTL3']
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.10
PRECO_MINIMO = 5.00
BANDA_ZONA_PCT = 0.01
EXIGIR_CONFLUENCIA_CANDLE = True
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
CACHE_MACRO_EXPIRY_HORAS = 24
ALTA_CONFIABILIDADE = False
DIST_CORDA_MAX = 30.0
USAR_GATILHO_BOLLINGER = False
USAR_GUARDIAO_MACD = False
MAX_ATIVOS_POR_SETOR = 2
MODO_GEBRA = 'black_belt'
LOG_DETALHADO_TICKER = True
LOG_PERFORMANCE = True
LOG_FILTROS_DETALHADO = True
VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
EXIGIR_CONFLUENCIA = True
print("✅ Parâmetros v9.0 carregados")

In [ ]:
!pip install yfinance pandas-ta python-dotenv requests-cache requests-ratelimiter gspread oauth2client --quiet 2>/dev/null
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, os, sys, traceback, gc, subprocess
from typing import Optional, Tuple, Dict, List, Any
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from requests_cache import CachedSession
from requests_ratelimiter import LimiterSession
import gspread
from oauth2client.service_account import ServiceAccountCredentials
warnings.filterwarnings("ignore", category=FutureWarning, module="pandas")
try:
    from dotenv import load_dotenv
    load_dotenv()
    if not EMAIL_REMETENTE:
        EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
    if not SENHA_APP:
        SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')
except ImportError:
    pass
try:
    session = LimiterSession(
        per_second=0.4,
        session_factory=lambda: CachedSession(
            cache_name='yfinance.cache', backend='sqlite', expire_after=timedelta(hours=6)
        )
    )
    yf.shared._requests = session
    print("✅ Sessão YFinance com cache e rate limit ativada")
except Exception as e:
    print(f"⚠️ Falha ao configurar cache: {e}")
class Logger:
    def __init__(self, arquivo_log, arquivo_detalhado=None):
        self.arquivo_log = arquivo_log
        self.arquivo_detalhado = arquivo_detalhado
        self.inicio_geral = time.time()
        self.timings = {}
        self.contadores = {}
        self._buffer = []
        self._max_buffer_size = 100
    def log(self, mensagem, nivel="INFO", ticker=None):
        ts = datetime.now().strftime("%H:%M:%S")
        msg = f"[{ts}] [{nivel}] {mensagem}"
        if ticker:
            msg += f" | {ticker}"
        print(msg)
        if self.arquivo_detalhado and LOG_PERFORMANCE:
            self._buffer.append(msg + "\n")
            if len(self._buffer) >= self._max_buffer_size:
                self._flush_buffer()
    def _flush_buffer(self):
        if self.arquivo_detalhado and self._buffer:
            try:
                with open(self.arquivo_detalhado, 'a', encoding='utf-8') as f:
                    f.writelines(self._buffer)
                self._buffer.clear()
            except Exception as e:
                print(f"[ERRO] Falha ao gravar log: {e}")
    def limpar_buffer(self):
        self._flush_buffer()
        self._buffer.clear()
    def warn(self, mensagem, ticker=None):
        self.log(mensagem, "WARN", ticker)
    def error(self, mensagem, ticker=None):
        self.log(mensagem, "ERRO", ticker)
    def iniciar_etapa(self, nome):
        self.timings[nome] = {'inicio': time.time()}
        self.log(f"🚀 Iniciando: {nome}", "ETAPA")
    def concluir_etapa(self, nome, detalhes=None):
        if nome in self.timings:
            dur = time.time() - self.timings[nome]['inicio']
            self.timings[nome]['duracao'] = dur
            msg = f"✅ Concluído: {nome} ({dur:.2f}s)"
            if detalhes:
                msg += " | " + " | ".join(f"{k}: {v}" for k, v in detalhes.items())
            self.log(msg, "ETAPA")
    def incrementar(self, contador, valor=1):
        self.contadores[contador] = self.contadores.get(contador, 0) + valor
    def resumo_final(self):
        self._flush_buffer()
        total = time.time() - self.inicio_geral
        self.log("\n" + "=" * 60, "RESUMO")
        self.log(f"⏱️ Tempo total: {total:.2f}s", "RESUMO")
        for etapa, dados in self.timings.items():
            if 'duracao' in dados:
                pct = dados['duracao'] / total * 100 if total > 0 else 0
                self.log(f"   • {etapa}: {dados['duracao']:.2f}s ({pct:.1f}%)", "RESUMO")
        if self.contadores:
            self.log("\n🔢 Contadores:", "RESUMO")
            for cont, val in self.contadores.items():
                self.log(f"   • {cont}: {val}", "RESUMO")
        self.log("=" * 60 + "\n", "RESUMO")
        try:
            with open('resumo_execucao.json', 'w', encoding='utf-8') as f:
                json.dump({'timestamp': datetime.now().isoformat(), 'duracao_total_segundos': total, 'timings': {k: {kk: vv for kk, vv in v.items() if kk != 'inicio'} for k, v in self.timings.items()}, 'contadores': self.contadores}, f, indent=2, ensure_ascii=False)
        except Exception:
            pass
    def __del__(self):
        try:
            self._flush_buffer()
        except:
            pass
logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)
def _log_exc(contexto, excecao):
    try:
        msg = f"[{contexto}] {type(excecao).__name__}: {str(excecao)[:200]}"
        if 'logger' in globals() and logger is not None:
            logger.error(msg)
        else:
            print(f"[ERRO] {msg}")
        full_trace = traceback.format_exc()
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        try:
            with open('traceback_errors.log', 'a', encoding='utf-8') as f:
                f.write(f"\n{'='*60}\n")
                f.write(f"Timestamp: {timestamp}\n")
                f.write(f"Contexto: {contexto}\n")
                f.write(f"Erro: {str(excecao)}\n")
                f.write(full_trace)
        except:
            pass
    except:
        print(f"[ERRO CRÍTICO] Exceção em {contexto}: {str(excecao)[:100]}")
def conectar_google_sheets():
    try:
        from google.colab import userdata
        key_content = userdata.get('GCP_SERVICE_ACCOUNT_KEY')
        if not key_content:
            logger.warn("Chave GCP não configurada.")
            return None, None, None
        json_key = json.loads(key_content)
        scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
        creds = ServiceAccountCredentials.from_json_keyfile_dict(json_key, scope)
        client = gspread.authorize(creds)
        planilha_id = userdata.get('GOOGLE_SHEET_ID')
        planilha = client.open_by_key(planilha_id)
        aba_circuit = planilha.worksheet("CircuitBreaker")
        aba_ledger = planilha.worksheet("Ledger")
        aba_scanner = planilha.worksheet("Scanner")
        logger.log("✅ Conectado ao Google Sheets", "INFO")
        return aba_circuit, aba_ledger, aba_scanner
    except Exception as e:
        logger.warn(f"Google Sheets indisponível: {e}")
        return None, None, None
aba_circuit, aba_ledger, aba_scanner = conectar_google_sheets()
try:
    from google.colab import userdata
    TELEGRAM_TOKEN = userdata.get('TELEGRAM_TOKEN')
    TELEGRAM_CHAT_ID = userdata.get('TELEGRAM_CHAT_ID')
except:
    TELEGRAM_TOKEN = os.getenv('TELEGRAM_TOKEN', '')
    TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID', '')
def enviar_telegram(mensagem, parse_mode='HTML'):
    if not TELEGRAM_TOKEN or not TELEGRAM_CHAT_ID:
        logger.warn("Telegram não configurado.")
        return
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {'chat_id': TELEGRAM_CHAT_ID, 'text': mensagem, 'parse_mode': parse_mode}
    try:
        requests.post(url, data=payload, timeout=10)
    except Exception as e:
        _log_exc('Telegram', e)
logger.log("🔧 Sistema v9.0 inicializado", "INFO")
print("✅ Célula 1 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 2: FUNÇÕES AUXILIARES (VETORIZADAS E ORIGINAIS)
# =============================================================================
def _safe_divide(a, b, default=np.nan):
    if b is None or b == 0 or pd.isna(b): return default
    return a / b
def _safe_log(x, default=np.nan):
    if x is None or x <= 0 or pd.isna(x): return default
    return np.log(x)
def calcular_eficiencia_candle(df):
    corpo = abs(df['Close'] - df['Open'])
    sombra_sup = df['High'] - df[['Close', 'Open']].max(axis=1)
    sombra_inf = df[['Close', 'Open']].min(axis=1) - df['Low']
    range_total = df['High'] - df['Low']
    range_total = range_total.replace(0, np.nan)
    eficiencia = pd.Series(index=df.index, dtype=float)
    alta = df['Close'] > df['Open']
    baixa = df['Close'] < df['Open']
    eficiencia[alta] = 1 - (sombra_sup[alta] / range_total[alta])
    eficiencia[baixa] = 1 - (sombra_inf[baixa] / range_total[baixa])
    return eficiencia
def detectar_regime(df, janela=20):
    df_temp = df.copy()
    df_temp['retorno'] = df_temp['Close'].pct_change()
    df_temp['volatilidade'] = df_temp['retorno'].rolling(janela).std()
    try:
        adx = ta.adx(df_temp['High'], df_temp['Low'], df_temp['Close'], length=14)
        if adx is not None and not adx.empty:
            if isinstance(adx, pd.DataFrame) and 'ADX_14' in adx.columns:
                df_temp['adx'] = adx['ADX_14']
            else:
                df_temp['adx'] = adx.iloc[:, 0] if len(adx.shape) > 1 else adx
        else:
            return pd.Series(index=df.index, dtype=int)
    except Exception as e:
        _log_exc('detectar_regime', e)
        return pd.Series(index=df.index, dtype=int)
    df_temp = df_temp.dropna(subset=['volatilidade', 'adx'])
    if df_temp.empty:
        return pd.Series(index=df.index, dtype=int)
    v_p33, v_p67 = df_temp['volatilidade'].quantile([0.33, 0.67])
    a_p33, a_p67 = df_temp['adx'].quantile([0.33, 0.67])
    def classificar(row):
        v, a = row['volatilidade'], row['adx']
        if v < v_p33 and a < a_p33: return 0
        elif v > v_p67 or a > a_p67: return 2
        return 1
    regimes = df_temp.apply(classificar, axis=1)
    regime_series = pd.Series(index=df.index, dtype=int)
    regime_series.loc[regimes.index] = regimes
    regime_series.ffill(inplace=True)
    return regime_series
def detectar_swing_low(df, janela=10, confirmar=True):
    if len(df) < janela:
        return float(df['Low'].min())
    lows = df['Low']
    min_rolling = lows.rolling(window=2*janela+1, center=True).min()
    is_swing = (lows == min_rolling) & ~min_rolling.isna()
    if confirmar:
        close_shifted = df['Close'].shift(-1)
        is_swing = is_swing & (close_shifted > lows)
    swing_vals = lows[is_swing]
    return float(swing_vals.iloc[-1]) if len(swing_vals) > 0 else float(lows.min())
def detectar_swing_high(df, janela=10, confirmar=True):
    if len(df) < janela:
        return float(df['High'].max())
    highs = df['High']
    max_rolling = highs.rolling(window=2*janela+1, center=True).max()
    is_swing = (highs == max_rolling) & ~max_rolling.isna()
    if confirmar:
        close_shifted = df['Close'].shift(-1)
        is_swing = is_swing & (close_shifted < highs)
    swing_vals = highs[is_swing]
    return float(swing_vals.iloc[-1]) if len(swing_vals) > 0 else float(highs.max())
def calcular_lta_pivos(df, janela_pivo=5):
    lows = df['Low']
    if len(lows) < 2*janela_pivo+1:
        return None
    log_lows = np.log(lows.replace(0, np.nan))
    min_roll = log_lows.rolling(window=2*janela_pivo+1, center=True).min()
    pivo_mask = (log_lows == min_roll) & ~min_roll.isna()
    indices_pivos = np.where(pivo_mask)[0]
    if len(indices_pivos) < 2:
        return None
    x1, x2 = indices_pivos[-2], indices_pivos[-1]
    y1, y2 = log_lows.iloc[x1], log_lows.iloc[x2]
    if x2 == x1: return None
    incl = (y2 - y1) / (x2 - x1)
    return np.exp(y2 + incl * (len(log_lows) - 1 - x2))
def calcular_ltb_pivos(df, janela_pivo=5):
    highs = df['High']
    if len(highs) < 2*janela_pivo+1:
        return None
    log_highs = np.log(highs.replace(0, np.nan))
    max_roll = log_highs.rolling(window=2*janela_pivo+1, center=True).max()
    pivo_mask = (log_highs == max_roll) & ~max_roll.isna()
    indices_pivos = np.where(pivo_mask)[0]
    if len(indices_pivos) < 2:
        return None
    x1, x2 = indices_pivos[-2], indices_pivos[-1]
    y1, y2 = log_highs.iloc[x1], log_highs.iloc[x2]
    if x2 == x1 or y2 >= y1:
        return None
    incl = (y2 - y1) / (x2 - x1)
    return np.exp(y2 + incl * (len(log_highs) - 1 - x2))
def detectar_armadilha_lta(df, lta, banda_pct=0.01):
    if len(df) < 2 or lta is None or lta <= 0:
        return False, 0.0
    ult = df.iloc[-1]
    low, close, open_ = ult['Low'], ult['Close'], ult['Open']
    corpo = abs(close - open_)
    range_c = ult['High'] - low
    if range_c <= 0: return False, 0.0
    sombra_inf = min(close, open_) - low
    zona_inf = lta * (1 - banda_pct)
    if low < zona_inf and close >= zona_inf and sombra_inf >= 2 * corpo:
        return True, min(1.0, sombra_inf / range_c)
    return False, 0.0
def detectar_armadilha_ltb(df, ltb, banda_pct=0.01):
    if len(df) < 2 or ltb is None or ltb <= 0:
        return False, 0.0
    ult = df.iloc[-1]
    high, close, open_ = ult['High'], ult['Close'], ult['Open']
    corpo = abs(close - open_)
    range_c = high - ult['Low']
    if range_c <= 0: return False, 0.0
    sombra_sup = high - max(close, open_)
    zona_sup = ltb * (1 + banda_pct)
    if high > zona_sup and close <= zona_sup and sombra_sup >= 2 * corpo:
        return True, min(1.0, sombra_sup / range_c)
    return False, 0.0
def calcular_lta_adaptativo(df, janelas=None):
    if janelas is None: janelas = [20, 30, 40, 50]
    if len(df) < max(janelas): return None
    lows, closes = df['Low'].values, df['Close'].values
    melhor_score, melhor = -np.inf, None
    for janela in janelas:
        lta = calcular_lta_pivos(df, janela_pivo=janela)
        if lta is None: continue
        zona_inf = lta * (1 - BANDA_ZONA_PCT)
        zona_sup = lta * (1 + BANDA_ZONA_PCT)
        ini = max(0, len(lows) - max(2*janela, 50))
        score = (np.sum(lows[ini:] >= zona_inf) - 3 * np.sum(closes[ini:] < zona_inf) + 2 * np.sum((lows[ini:] >= zona_inf) & (lows[ini:] <= zona_sup)))
        if score > melhor_score:
            melhor_score, melhor = score, (lta, janela)
    return melhor
def calcular_ltb_adaptativo(df, janelas=None):
    if janelas is None: janelas = [20, 30, 40, 50]
    if len(df) < max(janelas): return None
    highs, closes = df['High'].values, df['Close'].values
    melhor_score, melhor = -np.inf, None
    for janela in janelas:
        ltb = calcular_ltb_pivos(df, janela_pivo=janela)
        if ltb is None: continue
        zona_inf = ltb * (1 - BANDA_ZONA_PCT)
        zona_sup = ltb * (1 + BANDA_ZONA_PCT)
        ini = max(0, len(highs) - max(2*janela, 50))
        score = (np.sum(highs[ini:] <= zona_sup) - 3 * np.sum(closes[ini:] > zona_sup) + 2 * np.sum((highs[ini:] >= zona_inf) & (highs[ini:] <= zona_sup)))
        if score > melhor_score:
            melhor_score, melhor = score, (ltb, janela)
    return melhor
def validar_elliott(df, swing_lows, swing_highs):
    if len(swing_lows) < 5 or len(swing_highs) < 5:
        return False, "Pivôs insuficientes"
    try:
        onda1_low = swing_lows[-5]
        onda1_high = swing_highs[-4]
        onda2_low = swing_lows[-4]
        onda3_high = swing_highs[-3]
        onda4_low = swing_lows[-3]
        onda5_high = swing_highs[-2]
        if onda2_low < onda1_low: return False, "Onda 2 inválida"
        if onda4_low < onda1_high: return False, "Onda 4 invadiu Onda 1"
        amp1 = abs(onda1_high - onda1_low)
        amp3 = abs(onda3_high - onda2_low)
        amp5 = abs(onda5_high - onda4_low)
        if amp3 <= min(amp1, amp5): return False, "Onda 3 não é a maior"
        return True, "1-2-3-4-5"
    except Exception as e:
        _log_exc('validar_elliott', e)
        return False, "Erro na validação"
def calcular_obv_divergencia(df):
    if len(df) < 20: return None
    try:
        obv = ta.obv(df['Close'], df['Volume'])
        if obv is None or len(obv) < 20: return None
        preco_rec = df['Close'].iloc[-20:].values
        obv_rec = obv.iloc[-20:].values
        if preco_rec[-1] < preco_rec[0] and obv_rec[-1] > obv_rec[0]: return 'alta'
        if preco_rec[-1] > preco_rec[0] and obv_rec[-1] < obv_rec[0]: return 'baixa'
        return None
    except Exception as e:
        _log_exc('calcular_obv_divergencia', e)
        return None
def calcular_willr(df, periodo=14):
    try:
        willr = ta.willr(df['High'], df['Low'], df['Close'], length=periodo)
        if willr is not None and not willr.empty:
            val = willr.iloc[-1]
            return round(float(val), 1) if pd.notna(val) else None
    except Exception as e:
        _log_exc('calcular_willr', e)
    return None
def calcular_rsi(df, periodo=14):
    try:
        rsi = ta.rsi(df['Close'], length=periodo)
        if rsi is not None and not rsi.empty:
            val = rsi.iloc[-1]
            return round(float(val), 1) if pd.notna(val) else None
    except Exception as e:
        _log_exc('calcular_rsi', e)
    return None
def calcular_macd(df):
    try:
        macd = ta.macd(df['Close'])
        if macd is not None and not macd.empty:
            m = macd['MACD_12_26_9'].iloc[-1]
            s = macd['MACDs_12_26_9'].iloc[-1]
            h = macd['MACDh_12_26_9'].iloc[-1]
            return (round(float(m), 2) if pd.notna(m) else None,
                    round(float(s), 2) if pd.notna(s) else None,
                    round(float(h), 2) if pd.notna(h) else None)
    except Exception as e:
        _log_exc('calcular_macd', e)
    return None, None, None
def calcular_estocastico(df, periodo=14):
    try:
        stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=periodo, d=3)
        if stoch is not None and not stoch.empty:
            k = stoch['STOCHk_14_3_3'].iloc[-1]
            d = stoch['STOCHd_14_3_3'].iloc[-1]
            return (round(float(k), 1) if pd.notna(k) else None,
                    round(float(d), 1) if pd.notna(d) else None)
    except Exception as e:
        _log_exc('calcular_estocastico', e)
    return None, None
def calcular_bandas_bollinger(df, periodo=20):
    try:
        bb = ta.bbands(df['Close'], length=periodo)
        if bb is not None and not bb.empty:
            upper = float(bb['BBU_20_2.0'].iloc[-1])
            mid = float(bb['BBM_20_2.0'].iloc[-1])
            lower = float(bb['BBL_20_2.0'].iloc[-1])
            close = float(df['Close'].iloc[-1])
            pos = (close - lower) / (upper - lower) * 100 if upper != lower else 50
            return {'upper': round(upper, 2), 'mid': round(mid, 2), 'lower': round(lower, 2), 'posicao_%': round(pos, 1)}
    except Exception as e:
        _log_exc('calcular_bandas_bollinger', e)
    return None
def calcular_climax_volume(df, periodo=50):
    if len(df) < periodo: return False
    try:
        vol_med = df['Volume'].rolling(periodo).mean().iloc[-1]
        vol_at = df['Volume'].iloc[-1]
        return vol_at > 3 * vol_med if (pd.notna(vol_med) and vol_med > 0) else False
    except Exception as e:
        _log_exc('calcular_climax_volume', e)
        return False
def analisar_candle(row, row_ant=None):
    open_, high, low, close = row['Open'], row['High'], row['Low'], row['Close']
    corpo = abs(close - open_)
    range_total = high - low
    if range_total <= 0: return {}
    p_sup = high - max(open_, close)
    p_inf = min(open_, close) - low
    res = {}
    if p_inf >= 2*corpo and p_sup <= 0.3*range_total and corpo > 0: res['martelo'] = True
    if p_sup >= 2*corpo and p_inf <= 0.3*range_total and corpo > 0: res['estrela_cadente'] = True
    if corpo <= 0.05 * range_total: res['doji'] = True
    if row_ant is not None:
        open_ant, close_ant = row_ant['Open'], row_ant['Close']
        corpo_ant = abs(close_ant - open_ant)
        if corpo < corpo_ant and high <= row_ant['High'] and low >= row_ant['Low']:
            if close_ant < open_ant and close > open_: res['harami_alta'] = True
            elif close_ant > open_ant and close < open_: res['harami_baixa'] = True
        if corpo > corpo_ant:
            if close_ant < open_ant and close > open_ and open_ <= close_ant and close >= open_ant: res['engolfo_alta'] = True
            if close_ant > open_ant and close < open_ and open_ >= close_ant and close <= open_ant: res['engolfo_baixa'] = True
        if close_ant < open_ant and close > open_ and open_ > close_ant: res['kicker_alta'] = True
        if close_ant > open_ant and close < open_ and open_ < close_ant: res['kicker_baixa'] = True
    return res
def detectar_bebe_abandonado(df):
    if len(df) < 4: return None
    c1, c2, c3 = df.iloc[-4], df.iloc[-3], df.iloc[-2]
    if c1['High'] <= 0 or c2['High'] <= 0 or c2['Low'] <= 0 or c3['Low'] <= 0: return None
    try:
        if c1['Close'] < c1['Open']:
            gap1 = (c2['Low'] - c1['High']) / c1['High']
        else:
            gap1 = (c1['Low'] - c2['High']) / c2['High']
        if c3['Close'] > c3['Open']:
            gap2 = (c3['High'] - c2['Low']) / c2['Low']
        else:
            gap2 = (c2['High'] - c3['Low']) / c3['Low']
    except:
        return None
    if gap1 > 0.02 and gap2 > 0.02 and analisar_candle(c2).get('doji'):
        if c1['Close'] < c1['Open'] and c3['Close'] > c3['Open']:
            return {'tipo': 'Bebe_abandonado_alta'}
        if c1['Close'] > c1['Open'] and c3['Close'] < c3['Open']:
            return {'tipo': 'Bebe_abandonado_baixa'}
    return None
def detectar_retangulo(df, janela=20):
    if len(df) < janela: return None
    highs, lows = df['High'].iloc[-janela:], df['Low'].iloc[-janela:]
    resist, suporte = highs.max(), lows.min()
    if resist - suporte < 0.02 * suporte: return None
    toques_resist = np.sum(highs.values >= resist * 0.99)
    toques_suporte = np.sum(lows.values <= suporte * 1.01)
    if toques_resist >= 2 and toques_suporte >= 2:
        return {'tipo': 'Retangulo', 'suporte': round(suporte, 2), 'resistencia': round(resist, 2)}
    return None
def detectar_alargamento(df, janela=20):
    if len(df) < janela: return None
    highs = df['High'].iloc[-janela:].values
    lows = df['Low'].iloc[-janela:].values
    if highs[-1] > highs[0] and lows[-1] < lows[0]:
        return {'tipo': 'Alargamento'}
    return None
def detectar_estrutura_dow(df):
    if len(df) < 26: return None
    ult_top, prev_top = df['High'].iloc[-1], df['High'].iloc[-26]
    ult_fnd, prev_fnd = df['Low'].iloc[-1], df['Low'].iloc[-26]
    if ult_top > prev_top and ult_fnd > prev_fnd:
        return {'tendencia_dow': 'ALTA'}
    elif ult_top < prev_top and ult_fnd < prev_fnd:
        return {'tendencia_dow': 'BAIXA'}
    return {'tendencia_dow': 'LATERAL'}
def calcular_fibonacci_retracao(df):
    if len(df) < 50: return None
    sh = detectar_swing_high(df, janela=10, confirmar=False)
    sl = detectar_swing_low(df, janela=10, confirmar=False)
    if sh is None or sl is None or sh <= sl or sl <= 0: return None
    diff = sh - sl
    return {'38.2%': round(sh - diff * 0.382, 2), '50.0%': round(sh - diff * 0.500, 2), '61.8%': round(sh - diff * 0.618, 2)}
def calcular_alvos_fibonacci(df, direcao, fib_window=20):
    if len(df) < fib_window: return {}
    try:
        df_rec = df.iloc[-fib_window:]
        sl = detectar_swing_low(df_rec, janela=5, confirmar=False)
        sh = detectar_swing_high(df_rec, janela=5, confirmar=False)
        if sl is None or sh is None or sl <= 0 or sh <= 0 or sl >= sh: return {}
        amp = np.log(sh) - np.log(sl)
        base = np.log(sh)
        mults = [1.000, 1.618, 2.618, 4.236]
        if direcao == 'COMPRA':
            return {f'{m*100}%': round(np.exp(base + amp * m), 2) for m in mults}
        return {f'{m*100}%': round(np.exp(base - amp * m), 2) for m in mults}
    except Exception as e:
        _log_exc('calcular_alvos_fibonacci', e)
        return {}
def calcular_payoff_real(entrada, alvo, stop, custos_pct, direcao='COMPRA'):
    if direcao == 'COMPRA':
        if stop >= entrada or alvo <= entrada: return 0.0
        risco = entrada - stop
        retorno_bruto = alvo - entrada
    elif direcao == 'VENDA':
        if stop <= entrada or alvo >= entrada: return 0.0
        risco = stop - entrada
        retorno_bruto = entrada - alvo
    else:
        return 0.0
    if risco <= 0: return 0.0
    custo_total = (entrada + alvo) * custos_pct
    retorno_liquido = max(0, retorno_bruto - custo_total)
    return round(retorno_liquido / risco, 2)
def detectar_regime_volatilidade(serie, janela=40):
    try:
        if serie is None or serie.empty or len(serie) < janela: return 'BAIXA'
        ret = serie.pct_change().dropna()
        if ret.empty or len(ret) < 20: return 'BAIXA'
        vol_at = ret.rolling(20).std().iloc[-1]
        if hasattr(vol_at, 'iloc'): vol_at = vol_at.iloc[0]
        vol_hist = ret.rolling(janela).std().dropna()
        if vol_hist.empty: return 'BAIXA'
        proporcao = (vol_hist < vol_at).mean()
        return 'ALTA' if proporcao > 0.7 else 'BAIXA'
    except Exception as e:
        _log_exc('detectar_regime_volatilidade', e)
        return 'BAIXA'
def detectar_volume_anormal(df, periodo=20, limiar=1.5):
    if len(df) < periodo: return False
    try:
        vol_medio = df['Volume'].rolling(periodo).mean().iloc[-1]
        vol_atual = df['Volume'].iloc[-1]
        return vol_atual >= vol_medio * limiar if (pd.notna(vol_medio) and vol_medio > 0) else False
    except Exception as e:
        _log_exc('detectar_volume_anormal', e)
        return False
def fractional_kelly(win_rate, payoff_ratio, frac=0.25):
    if payoff_ratio <= 0: return 0.0
    kelly = (payoff_ratio * win_rate - (1 - win_rate)) / payoff_ratio
    return max(0.0, min(kelly, 0.25)) * frac
SENTIMENTO_PADRAO = 0.0
print("✅ Célula 2 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 3: GUARDIÕES E ANÁLISE (MANTIDOS DA v8.5.3)
# =============================================================================
SETOR_POR_TICKER = {
    'PETR4': 'Petróleo', 'PETR3': 'Petróleo', 'PRIO3': 'Petróleo',
    'VALE3': 'Mineração', 'GGBR4': 'Siderurgia', 'CSNA3': 'Siderurgia',
    'ITUB4': 'Financeiro', 'BBDC4': 'Financeiro', 'BBAS3': 'Financeiro',
    'ABEV3': 'Consumo', 'MGLU3': 'Varejo', 'RENT3': 'Varejo',
    'WEGE3': 'Indústria', 'RADL3': 'Saúde', 'JBSS3': 'Alimentos',
}
def obter_setor(ticker):
    return SETOR_POR_TICKER.get(ticker.replace('.SA', ''), 'Outros')

MACRO_REFERENCE = {
    'VALE3.SA': ('GC=F', 0.6),
    'PETR4.SA': ('CL=F', 0.8),
    'PETR3.SA': ('CL=F', 0.8),
    'ABEV3.SA': ('CORN', 0.3)
}
_cache_macro_data, _cache_macro_timestamp = {}, {}

def _obter_dados_macro(bench):
    agora = datetime.now()
    if bench in _cache_macro_timestamp:
        if (agora - _cache_macro_timestamp[bench]).total_seconds() / 3600 < CACHE_MACRO_EXPIRY_HORAS:
            return _cache_macro_data.get(bench)
    try:
        df_b = yf.download(bench, period='1y', interval='1wk', progress=False, auto_adjust=True)
        vals = df_b['Close'].values if not df_b.empty else None
        _cache_macro_data[bench] = vals
        _cache_macro_timestamp[bench] = agora
        return vals
    except Exception as e:
        _log_exc('_obter_dados_macro', e)
        return None

def verificar_alinhamento_macro(ticker, direcao, _=None):
    if ticker not in MACRO_REFERENCE:
        return True, 15
    ref, _ = MACRO_REFERENCE[ticker]
    precos = _obter_dados_macro(ref)
    if precos is None or len(precos) < 150:
        return True, 15
    ema50 = pd.Series(precos).ewm(span=50, adjust=False).mean()
    e_at, e_lag = ema50.iloc[-1], ema50.iloc[-5]
    if pd.isna(e_at) or pd.isna(e_lag):
        return True, 15
    slope_pos = e_at > e_lag
    if direcao == 'COMPRA' and precos[-1] > e_at and slope_pos:
        return True, 30
    if direcao == 'VENDA' and precos[-1] < e_at and not slope_pos:
        return True, 30
    return False, 0

def validar_toque_zona_wyckoff(df, lta, banda_pct=0.01, volume_min_ratio=0.8):
    if len(df) < 2 or lta is None or lta <= 0:
        return False, 'indefinido'
    ult = df.iloc[-1]
    low, close = ult['Low'], ult['Close']
    vol_atual = ult['Volume']
    vol_med = df['Volume'].rolling(20).mean().iloc[-1] if len(df) >= 20 else vol_atual
    zona_inf = lta * (1 - banda_pct)
    zona_sup = lta * (1 + banda_pct)
    toque_valido = (low <= zona_sup) and (close >= zona_inf)
    if not toque_valido:
        return False, 'indefinido'
    suporte_range = df['Low'].rolling(20).min().iloc[-1]
    resistencia_range = df['High'].rolling(20).max().iloc[-1]
    range_total = resistencia_range - suporte_range
    posicao_no_range = (close - suporte_range) / range_total if range_total > 0 else 0.5
    if posicao_no_range < 0.4:
        regime = 'acumulacao'
        toque_valido = vol_atual >= vol_med * volume_min_ratio if (pd.notna(vol_med) and vol_med > 0) else True
    elif posicao_no_range > 0.6:
        regime = 'markup'
        toque_valido = vol_atual >= vol_med * 1.2 if (pd.notna(vol_med) and vol_med > 0) else True
    else:
        regime = 'neutro'
        toque_valido = vol_atual >= vol_med * 0.8 if (pd.notna(vol_med) and vol_med > 0) else True
    return toque_valido, regime

def detectar_fase_wyckoff_adaptativo(df_w, suporte=None, resistencia=None, atr_period=14):
    if len(df_w) < 30:
        return 'INDEFINIDO', 0.0
    try:
        atr = ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=atr_period)
        if atr is None or atr.empty:
            return 'INDEFINIDO', 0.0
        atr_med = atr.rolling(20).mean()
        atr_val, atr_med_val = atr.iloc[-1], atr_med.iloc[-1]
        vol_rel = atr_val / atr_med_val if (pd.notna(atr_med_val) and atr_med_val > 0) else 1.0
    except Exception:
        return 'INDEFINIDO', 0.0
    thr_base = PARAMS_ATIVOS.get('wyckoff_threshold', 0.75)
    thr_comp = max(0.65, min(0.90, thr_base + 0.10 * (vol_rel - 1)))
    range_s = df_w['High'] - df_w['Low']
    r_med = range_s.rolling(20).mean().iloc[-1]
    if range_s.iloc[-1] >= r_med * thr_comp:
        return 'INDEFINIDO', 0.0
    px = df_w['Close'].iloc[-1]
    suporte = suporte if suporte else df_w['Low'].rolling(20).min().iloc[-1]
    resistencia = resistencia if resistencia else df_w['High'].rolling(20).max().iloc[-1]
    faixa = resistencia - suporte
    if faixa <= 0:
        return 'INDEFINIDO', 0.0
    pos_rel = (px - suporte) / faixa
    candles = df_w.iloc[-8:]
    alta = candles['Close'] > candles['Open']
    baixa = candles['Close'] < candles['Open']
    vol_alta = candles.loc[alta, 'Volume'].mean() if alta.any() else 0
    vol_baixa = candles.loc[baixa, 'Volume'].mean() if baixa.any() else 0
    if vol_baixa == 0:
        return 'INDEFINIDO', 0.0
    razao = vol_alta / vol_baixa
    conf = min(1.0, (r_med - range_s.iloc[-1]) / r_med) if r_med > 0 else 0.5
    if pos_rel < 0.4 and razao > 1.2:
        return 'ACUMULACAO', conf
    if pos_rel > 0.6 and razao < 0.8:
        return 'DISTRIBUICAO', conf
    return 'INDEFINIDO', 0.0

def _normalizar_dataframe(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(col).strip() for col in df.columns.values]
    rename_map = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
    lowercase = {str(c).lower(): c for c in df.columns}
    for lower, original in lowercase.items():
        if lower in rename_map:
            df.rename(columns={original: rename_map[lower]}, inplace=True)
    df.sort_index(inplace=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def _safe_atr(df_high, df_low, df_close, length):
    try:
        atr_series = ta.atr(df_high, df_low, df_close, length=length)
        if atr_series is None or atr_series.empty:
            return 0.0
        val = atr_series.iloc[-1]
        return float(val) if pd.notna(val) else 0.0
    except Exception as e:
        _log_exc('_safe_atr', e)
        return 0.0

# Guardiões
def guardiao_dow(df_w, contexto_trap, modo_gebra):
    if modo_gebra != 'black_belt': return True, None
    dow = detectar_estrutura_dow(df_w)
    ok = contexto_trap or (dow and dow['tendencia_dow'] == 'ALTA')
    return ok, 'Dow' if not ok else None

def guardiao_elliott(df_w, contexto_trap, modo_gebra):
    if modo_gebra != 'black_belt': return True, None, False
    swing_highs, swing_lows = [], []
    for j in range(5, len(df_w)-5):
        if df_w['High'].values[j] >= max(df_w['High'].values[j-5:j+6]):
            swing_highs.append(df_w['High'].values[j])
        if df_w['Low'].values[j] <= min(df_w['Low'].values[j-5:j+6]):
            swing_lows.append(df_w['Low'].values[j])
    valido, _ = validar_elliott(df_w, swing_lows, swing_highs)
    ok = contexto_trap or valido
    return ok, 'Elliott' if not ok else None, valido

def guardiao_fibonacci(df_w, entrada):
    fib_ret = calcular_fibonacci_retracao(df_w)
    if fib_ret:
        ok = (fib_ret['61.8%'] <= entrada <= fib_ret['38.2%'])
        return ok, 'Fibonacci' if not ok else None
    return True, None

def guardiao_retangulo(df_w, entrada, modo_gebra):
    if modo_gebra != 'black_belt': return True, None
    ret = detectar_retangulo(df_w)
    if ret and 'suporte' in ret:
        ok = entrada <= ret['suporte'] * 1.05
        return ok, 'Retângulo' if not ok else None
    return True, None

def guardiao_estocastico(df_w, modo_gebra):
    if modo_gebra != 'black_belt': return True, None
    stoch_k, _ = calcular_estocastico(df_w)
    ok = stoch_k is not None and stoch_k < 30
    return ok, 'Estocástico' if not ok else None

def guardiao_medias(df_w, entrada, modo_gebra):
    if modo_gebra != 'black_belt': return True, None
    mm200w = df_w['Close'].rolling(200).mean().iloc[-1]
    mm200w_ant = df_w['Close'].rolling(200).mean().iloc[-5] if len(df_w) >= 200 else mm200w
    ok = pd.notna(mm200w) and entrada > mm200w and mm200w > mm200w_ant
    return ok, 'Médias' if not ok else None

def guardiao_zona_wyckoff(df_w, lta, banda_pct):
    toca, regime = validar_toque_zona_wyckoff(df_w, lta, banda_pct)
    return toca, 'Zona Wyckoff' if not toca else None, regime

def guardiao_gatilho(df_w):
    pc = analisar_candle(df_w.iloc[-1], df_w.iloc[-2] if len(df_w) >= 2 else None)
    ok = pc.get('martelo') or pc.get('engolfo_alta') or pc.get('kicker_alta') or pc.get('harami_alta')
    return ok, 'Gatilho' if not ok else None, pc

def guardiao_payoff(entrada, alvo, stop, custos_pct, direcao='COMPRA', minimo=3.0):
    p_real = calcular_payoff_real(entrada, alvo, stop, custos_pct, direcao)
    ok = p_real >= minimo
    return ok, 'Payoff' if not ok else None, p_real

def guardiao_corda(df_w, entrada, modo_gebra, dist_max=30.0):
    if modo_gebra != 'black_belt': return True, None
    mm200w_val = df_w['Close'].rolling(200).mean().iloc[-1]
    if pd.notna(mm200w_val) and mm200w_val > 0:
        dist = (entrada - mm200w_val) / mm200w_val * 100
        ok = dist <= dist_max
        return ok, 'Corda' if not ok else None
    return True, None

def guardiao_macd(df_w, usar_macd):
    if not usar_macd: return True, None
    try:
        macd_calc = ta.macd(df_w['Close'])
        if macd_calc is None or macd_calc.empty: return True, None
        macd_line = macd_calc.iloc[:, 0].iloc[-1]
        signal_line = macd_calc.iloc[:, 1].iloc[-1]
        ok = macd_line > signal_line
        return ok, 'MACD' if not ok else None
    except Exception as e:
        _log_exc('guardiao_macd', e)
        return True, None

def guardiao_setor(ticker, contagem_setores, max_por_setor):
    setor = obter_setor(ticker)
    ok = contagem_setores.get(setor, 0) < max_por_setor
    return ok, f'Setor ({setor})' if not ok else None, setor

def calcular_score_qualidade(r, dow_ok, elliott_valido, fib_ok, ret_ok, stoch_ok, ma_ok, toca_zona, gatilho_ok, payoff_ok, corda_ok, alta_conf_ok, bollinger_ok, contexto_trap, is_arm, eficiencia, regime_wyckoff):
    score = 50
    if contexto_trap: score += 20
    if is_arm: score += 10
    if eficiencia and eficiencia > 0.8: score += 15
    elif eficiencia and eficiencia > 0.6: score += 8
    if elliott_valido: score += 10
    if dow_ok and not contexto_trap: score += 5
    if regime_wyckoff == 'acumulacao': score += 5
    elif regime_wyckoff == 'markup': score += 3
    if payoff_ok and r.get('Payoff Real', 0) > 4.0: score += 5
    return min(100, max(0, score))

logger.log("✅ Guardiões v9.0 carregados", "INFO")

In [ ]:
# =============================================================================
# CÉLULA 4: EXECUÇÃO PRINCIPAL COMPLETA (COM TODAS AS DEFINIÇÕES)
# =============================================================================

# Funções que estavam ausentes
def obter_tickers_b3() -> List[str]:
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE, 'r', encoding='utf-8') as f:
                cache = json.load(f)
            if (datetime.now() - datetime.fromisoformat(cache['timestamp'])).total_seconds() / 3600 < 24:
                logger.log(f"📦 Cache tickers ({len(cache['tickers'])} ativos)", "INFO")
                return cache['tickers']
        except:
            pass
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get("https://www.dadosdemercado.com.br/acoes", timeout=10, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        tickers = []
        for row in soup.select('table tbody tr'):
            cells = row.find_all('td')
            if cells and not cells[0].text.strip().startswith('#'):
                ticker = cells[0].text.strip().replace('.SA', '')
                if ticker:
                    tickers.append(ticker)
        if tickers:
            with open(CACHE_TICKERS_FILE, 'w', encoding='utf-8') as f:
                json.dump({'timestamp': datetime.now().isoformat(), 'tickers': tickers}, f)
            logger.log(f"🌐 Scraping ({len(tickers)} ativos)", "INFO")
            return tickers
    except Exception as e:
        logger.log(f"⚠️ Scraping falhou: {str(e)[:80]}", "WARN")
    logger.log("🔄 Fallback tickers", "WARN")
    return FALLBACK_TICKERS.copy()

def extrair_dataframe_ticker(data_raw, ticker: str) -> Optional[pd.DataFrame]:
    try:
        if isinstance(data_raw.columns, pd.MultiIndex):
            if (ticker not in data_raw.columns.get_level_values(1) and 
                ticker not in data_raw.columns.get_level_values(0)):
                return None
            df = data_raw[ticker].copy()
        else:
            df = data_raw.copy()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ['_'.join(col).strip() for col in df.columns.values]
        df.columns = [c.lower() for c in df.columns]
        rename_map = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
        df.rename(columns={k: rename_map.get(k, k) for k in df.columns if k in rename_map}, inplace=True)
        return df
    except Exception as e:
        _log_exc(f'extrair_dataframe_ticker {ticker}', e)
        return None

def resample_tf(df: pd.DataFrame, freq: str, min_days: int = 4, min_days_monthly: int = 10) -> Optional[pd.DataFrame]:
    if df is None or df.empty:
        return None
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            return None
    hoje = datetime.now()
    dia_semana = hoje.weekday()
    hora_atual = hoje.hour
    if freq.startswith('W'):
        if dia_semana < 4 or (dia_semana == 4 and hora_atual < 18):
            ultima_sexta = df.index[df.index.dayofweek == 4]
            if len(ultima_sexta) > 0:
                df = df.loc[:ultima_sexta[-1]]
                if df.empty:
                    return None
    agg = {'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'}
    df_r = df.resample(freq, closed='right', label='right').agg(agg)
    if freq.startswith('W'):
        counts = df.resample(freq, closed='right', label='right').count()['Close']
        df_r = df_r[counts >= min_days]
    elif freq in ('ME', 'M'):
        counts = df.resample(freq, closed='right', label='right').count()['Close']
        df_r = df_r[counts >= min_days_monthly]
    df_r = df_r.replace([np.inf, -np.inf], np.nan).dropna()
    df_r = df_r[df_r['Close'] > 0]
    return df_r

# ---- FLUXO PRINCIPAL ----
tickers_processados = []
logger.iniciar_etapa("Coleta de Tickers")
tickers_b3 = obter_tickers_b3()
tickers_b3 = [t.replace('.SA', '') for t in tickers_b3]
logger.concluir_etapa("Coleta de Tickers", {'total': len(tickers_b3)})

tickers_yahoo = [t + ".SA" for t in tickers_b3]
tickers_liquidos = []
BATCH = 50

logger.iniciar_etapa("Filtro de Liquidez")
for i in range(0, len(tickers_yahoo), BATCH):
    batch = tickers_yahoo[i:i+BATCH]
    try:
        data_raw = yf.download(batch, period='3mo', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS:
                continue
            df = extrair_dataframe_ticker(data_raw, t)
            if df is None or df.empty or 'Volume' not in df.columns:
                continue
            try:
                vol_med = df['Volume'].rolling(21).mean().iloc[-1]
                preco = df['Close'].iloc[-1]
                if pd.isna(vol_med) or pd.isna(preco) or preco <= 0:
                    continue
                if (vol_med >= VOLUME_MINIMO_ACAO and (vol_med * preco) >= VOLUME_FINANCEIRO_MINIMO):
                    tickers_liquidos.append(t)
            except Exception as e:
                _log_exc(f'Filtro liquidez {t}', e)
                continue
    except Exception as e:
        logger.log(f"Erro baixando lote {i//BATCH}: {str(e)[:100]}", "ERRO")
    time.sleep(1)
if len(tickers_liquidos) < 10:
    logger.log("Poucos ativos, usando fallback", "WARN")
    tickers_liquidos = [t + ".SA" for t in FALLBACK_TICKERS[:20]]
logger.concluir_etapa("Filtro de Liquidez", {'liquidos': len(tickers_liquidos)})

# Ingestão inteligente (Brapi + fallback)
try:
    from google.colab import userdata
    BRAPI_TOKEN = userdata.get('BRAPI_API_TOKEN')
except:
    BRAPI_TOKEN = os.getenv('BRAPI_API_TOKEN', '')

BRAPI_BASE = "https://brapi.dev/api"

def get_historical_data_brapi(tickers, period='5y'):
    results = {}
    for i in range(0, len(tickers), 10):
        batch = tickers[i:i+10]
        symbols = ','.join([t.replace('.SA','') for t in batch])
        url = f"{BRAPI_BASE}/quote/{symbols}?range={period}&interval=1d&token={BRAPI_TOKEN}"
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code == 200:
                data = resp.json()
                for quote in data.get('results', []):
                    ticker = quote['symbol'] + '.SA'
                    df = pd.DataFrame(quote['historicalDataPrice'])
                    df['date'] = pd.to_datetime(df['date'], unit='s')
                    df.set_index('date', inplace=True)
                    df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'}, inplace=True)
                    results[ticker] = df
        except Exception as e:
            _log_exc('Brapi download', e)
            continue
    return results

def download_with_fallback(tickers_yahoo, period='5y'):
    if BRAPI_TOKEN:
        logger.log("📡 Tentando Brapi.dev", "INFO")
        data = get_historical_data_brapi(tickers_yahoo, period)
        if len(data) >= 0.8 * len(tickers_yahoo):
            logger.log(f"✅ Brapi retornou {len(data)} ativos", "INFO")
            return data
        logger.log("⚠️ Brapi parcial, complementando com yfinance", "WARN")
    logger.log("🔄 Fallback para yfinance (com cache)", "INFO")
    data_d = {}
    for i in range(0, len(tickers_yahoo), BATCH):
        batch = tickers_yahoo[i:i+BATCH]
        try:
            df_raw = yf.download(batch, period=period, interval='1d', group_by='ticker', progress=False, auto_adjust=True)
            for t in batch:
                df = extrair_dataframe_ticker(df_raw, t)
                if df is not None and not df.empty:
                    data_d[t] = df
        except Exception as e:
            _log_exc(f'YFinance batch', e)
        time.sleep(1)
    return data_d

logger.iniciar_etapa("Download Dados Históricos")
data_d = download_with_fallback(tickers_liquidos, period='5y')
logger.concluir_etapa("Download Dados", {'sucesso': len(data_d)})

logger.iniciar_etapa("Resample Semanal/Mensal")
data_w, data_m = {}, {}
for t in tickers_liquidos:
    try:
        if t in data_d and not data_d[t].empty:
            df_d = data_d[t].copy()
            data_w[t] = resample_tf(df_d, 'W-FRI')
            data_m[t] = resample_tf(df_d, 'ME', min_days_monthly=10)
    except Exception as e:
        _log_exc(f'Resample {t}', e)
        continue
logger.concluir_etapa("Resample", {'semanais': len(data_w), 'mensais': len(data_m)})

# Indicadores de mercado
def calcular_nh_nl_simplificado(tickers_list, data_w_dict):
    count, total = 0, 0
    for t in tickers_list:
        if t not in data_w_dict or data_w_dict[t] is None or data_w_dict[t].empty:
            continue
        try:
            mm50 = data_w_dict[t]['Close'].rolling(50).mean().iloc[-1]
            close = data_w_dict[t]['Close'].iloc[-1]
            if pd.notna(mm50) and close > mm50:
                count += 1
            total += 1
        except Exception:
            continue
    return round(count / total * 100, 1) if total > 0 else None

def calcular_lad(tickers_list, data_d_dict):
    avancos, declinios, total = 0, 0, 0
    for t in tickers_list:
        if t not in data_d_dict or data_d_dict[t] is None or data_d_dict[t].empty:
            continue
        try:
            close = data_d_dict[t]['Close'].iloc[-1]
            close_ant = data_d_dict[t]['Close'].iloc[-2]
            if close > close_ant:
                avancos += 1
            elif close < close_ant:
                declinios += 1
            total += 1
        except Exception:
            continue
    return {'avancos': avancos, 'declinios': declinios, 'total': total, 'saldo': avancos - declinios}

nh_nl = calcular_nh_nl_simplificado(tickers_liquidos, data_w)
lad = calcular_lad(tickers_liquidos, data_d)
logger.log(f"📊 NH‑NL: {nh_nl}% | LAD saldo: {lad['saldo'] if lad else 'N/A'}", "INFO")

logger.iniciar_etapa("Regime de Volatilidade")
ibov = None
for simbolo in ["^BVSP", "^IBOV", "BOVA11.SA"]:
    try:
        ibov_raw = yf.download(simbolo, period='3mo', interval='1d', progress=False)
        if not ibov_raw.empty and 'Close' in ibov_raw.columns:
            ibov = ibov_raw['Close'].dropna()
            if len(ibov) >= 60:
                break
    except:
        pass
if ibov is not None and len(ibov) >= 60:
    regime_vol = detectar_regime_volatilidade(ibov)
else:
    regime_vol = 'BAIXA'
PARAMS_ATIVOS.clear()
PARAMS_ATIVOS.update(PARAMS_ALTA_VOL if regime_vol == 'ALTA' else PARAMS_BAIXA_VOL)
logger.concluir_etapa("Regime", {'regime': regime_vol})

kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO, PAYOFF_ESTIMADO, PARAMS_ATIVOS['kelly_frac'])
risco_maximo = CAPITAL_TOTAL * kelly_pct

# Circuit Breaker (cloud ou local)
def verificar_circuit_breakers():
    if aba_circuit is not None:
        try:
            registros = aba_circuit.get_all_records()
            hoje_str = datetime.now().strftime('%Y-%m-%d')
            pnl_diario = sum(float(r.get('pnl_real', 0)) for r in registros if r.get('data') == hoje_str)
            perdas = 0
            for r in sorted(registros, key=lambda x: x.get('data', ''), reverse=True):
                if float(r.get('pnl_real', 0)) < 0:
                    perdas += 1
                else:
                    break
            if abs(pnl_diario) / CAPITAL_TOTAL >= DRAWDOWN_MAX_DIARIO:
                return False, f"Drawdown >= {DRAWDOWN_MAX_DIARIO*100:.1f}%"
            if perdas >= MAX_PERDAS_CONSECUTIVAS:
                return False, f"{perdas} perdas consecutivas"
            return True, None
        except Exception as e:
            logger.error(f"Circuit Breaker cloud falhou: {e}")
    # Fallback local
    if not os.path.exists(ARQUIVO_LOG):
        return True, None
    try:
        with open(ARQUIVO_LOG, 'r', encoding='utf-8') as f:
            logs = json.load(f)
        if not isinstance(logs, list):
            return False, "Log corrompido"
        hoje = datetime.now().date()
        trades = [l for l in logs if l.get('tipo') == 'TRADE_FECHADO' and datetime.fromisoformat(l['timestamp']).date() == hoje]
        pnl = sum(float(t['dados'].get('pnl_real', 0)) for t in trades)
        if abs(pnl) / CAPITAL_TOTAL >= DRAWDOWN_MAX_DIARIO:
            return False, f"Drawdown >= {DRAWDOWN_MAX_DIARIO*100:.1f}%"
        perdas = 0
        for t in sorted(trades, key=lambda x: x['timestamp'], reverse=True):
            if float(t['dados'].get('pnl_real', 0)) < 0:
                perdas += 1
            else:
                break
        if perdas >= MAX_PERDAS_CONSECUTIVAS:
            return False, f"{perdas} perdas consecutivas"
        return True, None
    except Exception as e:
        logger.error(f"Circuit Breaker: {e}")
        return False, f"Falha de segurança: {str(e)[:100]}"

pode, motivo = verificar_circuit_breakers()
if not pode:
    logger.log(f"🛑 CIRCUIT BREAKER ATIVADO: {motivo}", "ALERT")
    enviar_telegram(f"🛑 CIRCUIT BREAKER: {motivo}")
    raise SystemExit("Circuit Breaker ativado")

oportunidades_swing, oportunidades_position = [], []
status_ativos = []
contagem_setores = {}
preco_minimo = PARAMS_ATIVOS.get('preco_minimo', PRECO_MINIMO)
risco_max_pct = PARAMS_ATIVOS.get('risco_percentual_maximo', RISCO_PERCENTUAL_MAXIMO)
stats_filtros = {key:0 for key in ['total_analisados','setup_aprovado_swing','setup_aprovado_position','bloqueios_preco','bloqueios_risco','bloqueios_confluencia','bloqueios_mm200','bloqueios_volume','bloqueios_lta','bloqueios_dow','bloqueios_elliott','bloqueios_fib','bloqueios_ret','bloqueios_stoch','bloqueios_ma','bloqueios_zona_wyckoff','bloqueios_gatilho','bloqueios_payoff','bloqueios_corda','bloqueios_macd','bloqueios_setor']}

logger.iniciar_etapa("Análise de Setups")
for i, ticker in enumerate(tickers_liquidos):
    if LOG_FILTROS_DETALHADO and i % 20 == 0:
        logger.log(f"Progresso: {i+1}/{len(tickers_liquidos)}", "DEBUG")
    
    df_w = data_w.get(ticker)
    if df_w is None or df_w.empty:
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Sem dados semanais'})
        continue
    stats_filtros['total_analisados'] += 1
    vol_fin = None
    try:
        vfc = (df_w['Volume'] * df_w['Close']).rolling(20).mean()
        vol_fin = vfc.iloc[-1] if pd.notna(vfc.iloc[-1]) else None
    except:
        pass
    df_w_norm = _normalizar_dataframe(df_w)
    df_w_norm['Eficiencia'] = calcular_eficiencia_candle(df_w_norm)
    df_w_norm['Regime'] = detectar_regime(df_w_norm)
    ult = df_w_norm.iloc[-1]
    entrada = float(ult['Close'])
    if pd.isna(entrada) or entrada <= 0 or entrada < preco_minimo:
        stats_filtros['bloqueios_preco'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Preço inválido'}); continue
    rh = float(df_w_norm['High'].rolling(window=min(52, len(df_w_norm))).max().iloc[-1])
    rl = float(df_w_norm['Low'].rolling(window=min(52, len(df_w_norm))).min().iloc[-1])
    reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
    ef = round(float(ult['Eficiencia']), 2) if not pd.isna(ult['Eficiencia']) else None
    if reg not in [1, 2] or ef is None or ef < 0.6:
        stats_filtros['bloqueios_confluencia'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Confluência'}); continue
    mm200w = df_w_norm['Close'].rolling(200).mean().iloc[-1]
    if pd.notna(mm200w) and entrada < mm200w:
        stats_filtros['bloqueios_mm200'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'MM200'}); continue
    if PARAMS_ATIVOS.get('exigir_volume_anormal', False) and not detectar_volume_anormal(df_w_norm):
        stats_filtros['bloqueios_volume'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Volume'}); continue
    res_lta = calcular_lta_adaptativo(df_w_norm)
    if res_lta is None:
        stats_filtros['bloqueios_lta'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'LTA'}); continue
    lta_val = res_lta[0]
    alargamento_detectado = detectar_alargamento(df_w_norm)
    is_arm_baixa, _ = detectar_armadilha_lta(df_w_norm, rl, BANDA_ZONA_PCT)
    contexto_trap = is_arm_baixa and (alargamento_detectado is not None)
    atr = _safe_atr(df_w_norm['High'], df_w_norm['Low'], df_w_norm['Close'], 14) or entrada * 0.02
    stop_atr = entrada - 1.8 * atr
    swing_low_val = detectar_swing_low(df_w_norm, janela=12)
    stop_candidatos = [s for s in [stop_atr, swing_low_val] if s is not None and s > 0 and s < entrada]
    if not stop_candidatos:
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Stop inválido'}); continue
    stop_final = max(stop_candidatos)
    risco = entrada - stop_final
    if risco / entrada < RISCO_PERCENTUAL_MINIMO or risco / entrada > risco_max_pct:
        stats_filtros['bloqueios_risco'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Risco fora'}); continue
    alvo = entrada + risco * 3
    # Guardiões
    ok_dow, lbl_dow = guardiao_dow(df_w_norm, contexto_trap, MODO_GEBRA)
    if not ok_dow:
        stats_filtros['bloqueios_dow'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_dow}); continue
    ok_ell, lbl_ell, ell_valido = guardiao_elliott(df_w_norm, contexto_trap, MODO_GEBRA)
    if not ok_ell:
        stats_filtros['bloqueios_elliott'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_ell}); continue
    ok_fib, lbl_fib = guardiao_fibonacci(df_w_norm, entrada)
    if not ok_fib:
        stats_filtros['bloqueios_fib'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_fib}); continue
    ok_ret, lbl_ret = guardiao_retangulo(df_w_norm, entrada, MODO_GEBRA)
    if not ok_ret:
        stats_filtros['bloqueios_ret'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_ret}); continue
    ok_stoch, lbl_stoch = guardiao_estocastico(df_w_norm, MODO_GEBRA)
    if not ok_stoch:
        stats_filtros['bloqueios_stoch'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_stoch}); continue
    ok_ma, lbl_ma = guardiao_medias(df_w_norm, entrada, MODO_GEBRA)
    if not ok_ma:
        stats_filtros['bloqueios_ma'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_ma}); continue
    ok_zona, lbl_zona, regime_wyckoff = guardiao_zona_wyckoff(df_w_norm, lta_val, BANDA_ZONA_PCT)
    if not ok_zona:
        stats_filtros['bloqueios_zona_wyckoff'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_zona}); continue
    ok_gat, lbl_gat, padroes_candle = guardiao_gatilho(df_w_norm)
    if not ok_gat:
        stats_filtros['bloqueios_gatilho'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_gat}); continue
    ok_pay, lbl_pay, payoff_real = guardiao_payoff(entrada, alvo, stop_final, PARAMS_ATIVOS['custos_pct'], 'COMPRA')
    if not ok_pay:
        stats_filtros['bloqueios_payoff'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_pay}); continue
    ok_corda, lbl_corda = guardiao_corda(df_w_norm, entrada, MODO_GEBRA, DIST_CORDA_MAX)
    if not ok_corda:
        stats_filtros['bloqueios_corda'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_corda}); continue
    ok_macd, lbl_macd = guardiao_macd(df_w_norm, USAR_GUARDIAO_MACD)
    if not ok_macd:
        stats_filtros['bloqueios_macd'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_macd}); continue
    ok_setor, lbl_setor, setor = guardiao_setor(ticker, contagem_setores, MAX_ATIVOS_POR_SETOR)
    if not ok_setor:
        stats_filtros['bloqueios_setor'] += 1; status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_setor}); continue
    
    contagem_setores[setor] = contagem_setores.get(setor, 0) + 1
    fat_liq = min(1.0, vol_fin / LIMITE_LIQUIDEZ_FINANCEIRA) if (pd.notna(vol_fin) and vol_fin and vol_fin > 0) else 0.5
    lote_base = int(risco_maximo / risco) if risco > 0 else 0
    lote_aj = max(1, int(lote_base * fat_liq))
    score_qualidade = calcular_score_qualidade({}, True, ell_valido, True, True, True, True, True, True, True, True, True, True, contexto_trap, is_arm_baixa, ef, regime_wyckoff)
    padroes = [k for k, v in padroes_candle.items() if v] if padroes_candle else []
    setup = {
        'Ticker': ticker, 'Direcao': 'COMPRA', 'Entrada': round(entrada, 2),
        'Stop Loss': round(stop_final, 2), 'Alvo Recomendado': round(alvo, 2),
        'Payoff Real': payoff_real, 'Score Qualidade': score_qualidade,
        'Padrões Detectados': ', '.join(padroes) if padroes else 'Nenhum'
    }
    stats_filtros['setup_aprovado_swing'] += 1
    oportunidades_swing.append(setup)
    status_ativos.append({'Ticker': ticker, 'Status': '✅ APROVADO', 'Filtro': 'Nenhum'})

logger.concluir_etapa("Análise de Setups", {'analisados': stats_filtros['total_analisados'], 'aprovados': stats_filtros['setup_aprovado_swing']})
if oportunidades_swing:
    oportunidades_swing = sorted(oportunidades_swing, key=lambda x: x.get('Score Qualidade', 0), reverse=True)[:MAX_SETUPS_POR_DIA]

logger.log(f"🎯 Swing: {len(oportunidades_swing)} setups", "RESULTADO")
logger.log(f"   Kelly: {kelly_pct*100:.2f}% | Regime: {regime_vol} | Modo: {MODO_GEBRA}", "RESULTADO")

# ---- RELATÓRIO DETALHADO E ENVIO ----
def gerar_relatorio_detalhado(status_ativos, oportunidades, regime_vol, nh_nl, lad, kelly_pct):
    linhas = []
    linhas.append("RELATÓRIO DETALHADO DE ANÁLISE - GEBRA v9.0")
    linhas.append(f"Data: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
    linhas.append(f"Regime: {regime_vol} | NH-NL: {nh_nl}% | LAD: {lad['saldo'] if lad else 'N/A'}")
    linhas.append(f"Kelly: {kelly_pct*100:.2f}% | Oportunidades: {len(oportunidades)}")
    linhas.append("")
    linhas.append("--- STATUS DOS ATIVOS (COM MOTIVO DA RECUSA) ---")
    for s in status_ativos:
        motivo = s.get('Filtro', 'N/A')
        linhas.append(f"{s['Ticker']:12} | {s['Status']:15} | Motivo: {motivo}")
    linhas.append("")
    if oportunidades:
        linhas.append("--- OPORTUNIDADES APROVADAS ---")
        for op in oportunidades:
            linhas.append(f"{op['Ticker']} | Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f} | Payoff: {op.get('Payoff Real',0)}")
    else:
        linhas.append("Nenhuma oportunidade aprovada hoje.")
    linhas.append("\n--- FIM ---")
    return "\n".join(linhas)

def enviar_email_detalhado(assunto, corpo_texto):
    if not EMAIL_REMETENTE or not SENHA_APP:
        logger.warn("E-mail não configurado.")
        return
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_REMETENTE
        msg['To'] = EMAIL_REMETENTE
        msg['Subject'] = assunto
        msg.attach(MIMEText(corpo_texto, 'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(EMAIL_REMETENTE, SENHA_APP)
            server.send_message(msg)
        logger.log("📧 E-mail detalhado enviado", "INFO")
    except Exception as e:
        _log_exc('Envio de e-mail', e)

relatorio = gerar_relatorio_detalhado(status_ativos, oportunidades_swing, regime_vol, nh_nl, lad, kelly_pct)
with open('relatorio_detalhado.txt', 'w', encoding='utf-8') as f:
    f.write(relatorio)
assunto_email = f"📊 GEBRA v9.0 - {len(oportunidades_swing)} Ops - {datetime.now().strftime('%d/%m %H:%M')}"
enviar_email_detalhado(assunto_email, relatorio)

def formatar_para_telegram(oportunidades, regime_vol, kelly_pct):
    if not oportunidades:
        return f"📊 <b>GEBRA v9.0</b>\nNenhuma oportunidade hoje.\nRegime: {regime_vol} | Kelly: {kelly_pct*100:.1f}%"
    msg = f"🚀 <b>ALERTAS GEBRA - {datetime.now().strftime('%d/%m %H:%M')}</b>\n"
    msg += f"Regime: {regime_vol} | Kelly: {kelly_pct*100:.1f}%\n\n"
    for i, op in enumerate(oportunidades[:MAX_SETUPS_POR_DIA], 1):
        msg += f"{i}. <b>{op['Ticker']}</b> | Dir: {op['Direcao']}\n"
        msg += f"   Entrada: R$ {op['Entrada']:.2f} | Stop: R$ {op['Stop Loss']:.2f}\n"
        msg += f"   Alvo: R$ {op.get('Alvo Recomendado',0):.2f} | Payoff: {op.get('Payoff Real',0):.2f}\n"
        msg += f"   Score: {op.get('Score Qualidade',0)}\n\n"
    return msg

enviar_telegram(formatar_para_telegram(oportunidades_swing, regime_vol, kelly_pct))

# Push para GitHub
def push_para_github():
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_PAT')
        if not token:
            return
        repo_url = f"https://{token}@github.com/seu_usuario/seu_repo.git"
        subprocess.run(["git", "config", "--global", "user.name", "TraderBot"])
        subprocess.run(["git", "config", "--global", "user.email", "bot@trading.com"])
        subprocess.run(["git", "add", "."])
        subprocess.run(["git", "commit", "-m", f"Execução {datetime.now().isoformat()}"], check=False)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url])
        subprocess.run(["git", "push", "origin", "main"])
        logger.log("✅ Push no GitHub concluído", "INFO")
    except Exception as e:
        logger.warn(f"Falha no push: {e}")
push_para_github()

logger._flush_buffer()
gc.collect()
logger.resumo_final()
logger.log("✅ Sistema v9.0 concluído com sucesso", "SUCCESS")
print("\n✅ Execução otimizada concluída. Verifique e-mail e Telegram.")